# Optimizing Customer-Support RAG — Accuracy, Cost, Speed

Build a customer-support RAG pipeline over real product manuals, then **optimize** it —
refusing to accept any optimization that isn't measured.

The corpus is 746 pages of real manuals:

| Product | Manual | Pages |
|---|---|---|
| car | Tata Sierra EV Owner's Manual | 408 |
| tv | Samsung TV User Guide | 329 |
| mouse | Logitech MX Master 3 Manual | 9 |

### The three axes

| Axis | Metric used here | Why this one |
|---|---|---|
| Accuracy | `quote_recall@k` — did retrieval return the chunk containing the answer? | Generation cannot fix retrieval. If the answer isn't in context, no model rescues it. |
| Cost | $/1000 queries, split into embedding + prompt + completion | Prompt tokens dominate RAG spend, and prompt tokens are a *retrieval* decision. |
| Speed | p50 / p95 end-to-end latency, broken down by stage | p95 is what customers feel. Averages hide the reranker. |

### Rules this notebook follows

1. **Every claim is a printed number.** No "hybrid search improves recall" without the delta.
2. **Sweeps use free metrics.** Chunking, embedding model, and ANN parameters are scored
   with retrieval metrics against a labelled gold set — no LLM calls, so a sweep is
   seconds and cents, not minutes and dollars. LLM judging is used once, at the end,
   on the configs that survived.
3. **Optimizations are evaluated in the order of their ceiling**, not the order they
   appear in blog posts. Section 1 explains why that ordering matters here.

### Running this

Self-contained: the next cell checks its own prerequisites and locates the PDFs itself,
so the notebook runs from any working directory.

```
pip install litellm langchain-text-splitters sentence-transformers chromadb pymupdf rank_bm25 python-dotenv numpy
```

A `.env` with `GROQ_API_KEY` is the only credential needed. Run top to bottom; CPU is
fine, no GPU required.

**Budget.** A notebook that teaches cost discipline has no business being expensive to
run. Every LLM call goes through a metered wrapper with a hard ceiling of **$0.50** for
the entire notebook — it raises rather than overspending, so a runaway loop costs a cent,
not a bill. Section 0 prices the whole run up front (~$0.05 on current groq pricing) and
the last cell asserts the actual total came in under the ceiling. Expensive steps are
cached to disk on first run, so re-running costs nothing and reproduces the same numbers.

In [24]:
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import pymupdf
import litellm
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, CrossEncoder

load_dotenv()
litellm.drop_params = True

FILES = {
    "car":   ("Tata Sierra EV Owner's Manual", "sierra-ev.pdf"),
    "tv":    ("Samsung TV User Guide",         "BN81-28760D-160_EUG_ROPDVBAAH_ASIA_ENG_260313.0.pdf"),
    "mouse": ("Logitech MX Master 3 Manual",   "logitech-mx-master-3.pdf"),
}

# Locate data/manuals from wherever this notebook was started (repo root, rag_at_scale/, ...).
CANDIDATES = [Path.cwd() / "data/manuals", Path.cwd() / "rag_at_scale/data/manuals",
              Path.cwd().parent / "data/manuals"]
DATA = next((p for p in CANDIDATES if p.is_dir()), None)
assert DATA, "data/manuals not found. Tried:\n  " + "\n  ".join(str(p) for p in CANDIDATES)

MANUALS = [{"product": k, "label": label, "path": DATA / fname} for k, (label, fname) in FILES.items()]
missing = [m["path"].name for m in MANUALS if not m["path"].exists()]
assert not missing, f"missing PDFs in {DATA}: {missing}"

CACHE = DATA.parent.parent / "eval"    # LLM results are cached here; delete it to force a cold run
CACHE.mkdir(exist_ok=True)

GEN_MODEL   = "groq/openai/gpt-oss-120b"   # answers, judging, gold-set generation
CHEAP_MODEL = "groq/openai/gpt-oss-20b"      # routing + the escalation experiment

if not os.getenv("GROQ_API_KEY"):
    print("WARNING: GROQ_API_KEY not set — every LLM cell will fail")


def norm(s):
    """Whitespace- and case-insensitive form, for substring matching against PDF text."""
    return re.sub(r"\s+", " ", s).strip().lower()


def cached_json(name, build):
    """Disk-cache a build step. Re-running this notebook must not re-pay for LLM calls."""
    p = CACHE / name
    if p.exists():
        return json.loads(p.read_text())
    val = build()
    p.write_text(json.dumps(val, indent=2, ensure_ascii=False))
    return val


print("manuals:", DATA)
print("cache:  ", CACHE)

manuals: /home/koireader/Documents/Agents/learn-llm/rag_at_scale/data/manuals
cache:   /home/koireader/Documents/Agents/learn-llm/rag_at_scale/eval


### The budget meter

Cost is one of the three things this notebook optimizes, so it is measured on the
notebook itself, not just on the pipeline. Every LLM call in every section goes through
`llm()`, which accumulates spend and **refuses to make the call** once the ceiling is
hit. A metered wrapper is a habit worth copying into production: the alternative to a
hard ceiling is discovering the number on an invoice.

In [38]:
BUDGET_USD = 0.50
SPEND = {"usd": 0.0, "calls": 0}


def cost_of(resp):
    c = (getattr(resp, "_hidden_params", None) or {}).get("response_cost")
    if c is None:
        try:
            c = litellm.completion_cost(resp)
        except Exception:
            c = 0.0          # unpriced model: metered as free rather than crashing the notebook
    return float(c or 0.0)


RETRIES = 6   # free-tier TPM is 8000; a throttled call is billed nothing, so waiting is free


def llm(**kwargs):
    """Every LLM call in this notebook goes through here. Hard budget stop, not a warning."""
    if SPEND["usd"] >= BUDGET_USD:
        raise RuntimeError(f"budget exhausted: ${SPEND['usd']:.4f} spent over {SPEND['calls']} calls "
                           f"(ceiling ${BUDGET_USD}). Raise BUDGET_USD deliberately, or fix the loop.")
    for attempt in range(RETRIES):
        try:
            resp = litellm.completion(**kwargs)
        except litellm.RateLimitError as e:
            if attempt == RETRIES - 1:
                raise
            # Groq states its own wait ("Please try again in 6.3825s"). Trust it over a guess.
            m = re.search(r"try again in ([\d.]+)s", str(e))
            wait = float(m.group(1)) + 1.0 if m else 2.0 ** attempt
            print(f"[ratelimit] waiting {wait:.1f}s ({attempt + 1}/{RETRIES})")
            time.sleep(wait)
            continue
        SPEND["usd"] += cost_of(resp)
        SPEND["calls"] += 1
        return resp


def spend(tag=""):
    print(f"[spend] ${SPEND['usd']:.4f} of ${BUDGET_USD:.2f} over {SPEND['calls']} calls  {tag}")


spend("start")

[spend] $0.0000 of $0.50 over 0 calls  start


### Costing the run before running it

The ceiling stops a disaster; it does not tell you whether the plan fits. The workload is
known in advance — the call counts below are structural, not guesses — so price it from
`litellm`'s own table first. This is the same arithmetic you should do before any batch
job, and it takes one cell.

In [3]:
def price(model, tok_in, tok_out):
    m = litellm.model_cost[model]
    return tok_in * m["input_cost_per_token"] + tok_out * m["output_cost_per_token"]


PLAN = [   # (stage, calls, model, prompt tokens, completion tokens)
    ("gold-set generation", 30, GEN_MODEL,   1150,  80),
    ("LLM router bench",    24, CHEAP_MODEL,   60,   3),
    ("policy: always 70B",  12, GEN_MODEL,    750, 300),
    ("policy: always 8B",   12, CHEAP_MODEL,  750, 300),
    ("policy: escalation",  12, GEN_MODEL,    750, 300),
    ("LLM judge",           36, GEN_MODEL,    250,   5),
    ("demos + cache misses", 6, GEN_MODEL,    750, 300),
]

print(f"{'stage':24s} {'calls':>6s} {'est. $':>9s}")
est = 0.0
for stage, n, model, ti, to in PLAN:
    c = n * price(model, ti, to)
    est += c
    print(f"{stage:24s} {n:6d} {c:9.4f}")
print(f"{'ESTIMATED TOTAL':24s} {sum(n for _, n, *_ in PLAN):6d} {est:9.4f}")
print(f"\nceiling ${BUDGET_USD:.2f} -> {BUDGET_USD/est:.0f}x headroom. The gap is deliberate: it "
      f"absorbs retries\nand a few extra experiments without ever absorbing a runaway loop.")

assert est < BUDGET_USD, "the plan does not fit the budget before a single call is made"

stage                     calls    est. $
gold-set generation          30    0.0223
LLM router bench             24    0.0001
policy: always 70B           12    0.0082
policy: always 8B            12    0.0007
policy: escalation           12    0.0082
LLM judge                    36    0.0055
demos + cache misses          6    0.0041
ESTIMATED TOTAL             132    0.0489

ceiling $0.50 -> 10x headroom. The gap is deliberate: it absorbs retries
and a few extra experiments without ever absorbing a runaway loop.


## 1. First: audit what actually made it into the corpus

The instinct when someone says "make RAG more accurate" is to reach for chunk size, a
better embedding model, a reranker. All of those move recall by single-digit percentages.
Before spending anything there, measure the **ceiling**: how much of the source material
reached the index, and in what condition. Retrieval optimizations only compete for points
inside that ceiling.

Two checks, both free, both one `pymupdf` loop. The first one comes back clean on this
corpus. The second one does not.

### Check 1: page coverage

In [4]:
def load_pages(manual):
    doc = pymupdf.open(manual["path"])
    pages = [{"product": manual["product"], "page": i + 1, "text": p.get_text()}
             for i, p in enumerate(doc)]
    doc.close()
    return pages


ALL_PAGES = [pg for m in MANUALS for pg in load_pages(m)]

MIN_CHARS = 30   # below this a page carries no retrievable content (page number + header)

print(f"{'manual':40s} {'pages':>6s} {'with text':>10s} {'empty':>7s} {'coverage':>9s} {'chars':>9s}")
for m in MANUALS:
    pgs = [p for p in ALL_PAGES if p["product"] == m["product"]]
    withtext = [p for p in pgs if len(p["text"].strip()) >= MIN_CHARS]
    chars = sum(len(p["text"]) for p in pgs)
    print(f"{m['label']:40s} {len(pgs):6d} {len(withtext):10d} {len(pgs)-len(withtext):7d} "
          f"{len(withtext)/len(pgs):8.0%} {chars:9,d}")

TEXT_PAGES = [p for p in ALL_PAGES if len(p["text"].strip()) >= MIN_CHARS]
print(f"\nCorpus: {len(TEXT_PAGES)} of {len(ALL_PAGES)} pages carry extractable text "
      f"({len(TEXT_PAGES)/len(ALL_PAGES):.0%})")

manual                                    pages  with text   empty  coverage     chars
Tata Sierra EV Owner's Manual               408        403       5      99%   561,484
Samsung TV User Guide                       329        329       0     100%   437,057
Logitech MX Master 3 Manual                   9          9       0     100%     7,273

Corpus: 741 of 746 pages carry extractable text (99%)


Coverage is essentially total: a handful of section dividers carry no text, everything
else extracted. **This is the good case, and it is worth knowing you are in it** — when
this table instead shows a large empty fraction (scanned manuals, or PDFs whose type was
converted to vector outlines at export), no retriever tuning substitutes for the missing
text, and the fix is upstream: get the source file from the publisher, or OCR the pages.
That is a different project with a different budget, and this corpus does not need it.

Move on to the second check, which is the one that bites here.

### Check 2: is the extracted text actually usable?

Coverage says text came out. It says nothing about what condition it is in. Manuals are
typeset in justified columns, and justified columns hyphenate — which shows up in the
extracted stream as words split in half.

In [5]:
SOFT = "\u00ad"      # soft hyphen: inserted by the typesetter, invisible when rendered

for m in MANUALS:
    txt = "".join(p["text"] for p in ALL_PAGES if p["product"] == m["product"])
    hard = len(re.findall(r"[a-z]-\n[a-z]", txt))
    print(f"{m['label']:34s} soft-hyphen breaks {txt.count(SOFT):5d} | hard hyphen+newline {hard:4d}")

car_txt = "".join(p["text"] for p in ALL_PAGES if p["product"] == "car")
print("\nwhat those look like in the extracted stream:")
for mt in list(re.finditer(r"\w+" + SOFT + r"\n\w+", car_txt))[:4]:
    print("   ", repr(mt.group(0)))

Tata Sierra EV Owner's Manual      soft-hyphen breaks  2272 | hard hyphen+newline   95
Samsung TV User Guide              soft-hyphen breaks     0 | hard hyphen+newline    6
Logitech MX Master 3 Manual        soft-hyphen breaks     0 | hard hyphen+newline    1

what those look like in the extracted stream:
    've\xad\nhicle'
    'care\xad\nfully'
    'sys\xad\ntems'
    've\xad\nhicle'


Every one of those is a word the index will never match. `ve\xad\nhicle` tokenizes as
`ve` + `hicle`; a customer asking about the *vehicle* battery matches neither. It damages
all three retrieval paths at once:

- **BM25** — lexical matching is exact, so the fragments are simply different terms
- **Embeddings** — split words tokenize into junk subwords, degrading the chunk vector
- **The generator** — even on a successful retrieval, the quoted context reads as broken

The fix is one line, but note *which* line. Soft hyphens (`U+00AD`) are unambiguous: the
typesetter inserted them, so joining is always correct. A **hard** hyphen before a
newline is ambiguous — `cus-\ntomer` should join to `customer`, but `toll-\nfree` must
stay hyphenated, and telling them apart needs a dictionary. Fix the unambiguous case,
leave the ambiguous one, and count what you left behind rather than quietly guessing.

In [6]:
def clean_text(t):
    t = t.replace(SOFT + "\n", "").replace(SOFT, "")   # unambiguous: rejoin the split word
    return t                                            # ponytail: hard "-\n" left alone on purpose


def broken_words(t):
    return set(re.findall(r"\b\w{2,}" + SOFT + r"\n\w{2,}\b", t))


before = broken_words(car_txt)
after = broken_words(clean_text(car_txt))
print(f"words split by a soft hyphen: {len(before)} distinct -> {len(after)} after cleaning")

# The repair is verifiable on a real term: does "vehicle" exist as a whole word?
probe = "vehicle"
raw_hits = len(re.findall(rf"\b{probe}\b", car_txt, re.I))
clean_hits = len(re.findall(rf"\b{probe}\b", clean_text(car_txt), re.I))
print(f"occurrences of the whole word {probe!r}: {raw_hits} raw -> {clean_hits} cleaned "
      f"(+{clean_hits - raw_hits} recovered)")
assert clean_hits > raw_hits, "cleaning should recover matches, not remove them"

leftover = len(re.findall(r"[a-z]-\n[a-z]", car_txt))
print(f"\nleft alone: {leftover} hard hyphen+newline breaks (ambiguous without a dictionary)")

words split by a soft hyphen: 999 distinct -> 0 after cleaning
occurrences of the whole word 'vehicle': 1486 raw -> 1580 cleaned (+94 recovered)

left alone: 95 hard hyphen+newline breaks (ambiguous without a dictionary)


In [7]:
CORPUS_PAGES = [{**p, "text": clean_text(p["text"])} for p in TEXT_PAGES]

print(f"corpus: {len(CORPUS_PAGES)} pages, "
      f"{sum(len(p['text']) for p in CORPUS_PAGES):,} chars, soft hyphens removed")
print("Everything downstream indexes this cleaned text.")

corpus: 741 pages, 1,001,230 chars, soft hyphens removed
Everything downstream indexes this cleaned text.


## 2. A gold set, or none of the rest of this notebook means anything

Every optimization below is a claim of the form "config B beats config A". That claim
needs a labelled set. Hand-writing one is slow and biases toward questions you already
know the pipeline answers, so generate it — but **generated labels are worthless
unless verified**, so every item is checked against the source page before it is kept:

- sample pages stratified across the three products
- ask an LLM for a customer-style question plus a **verbatim quote** from that page that
  answers it
- `assert` the quote actually appears in that page's text; discard the item if it doesn't

The quote is the label. Retrieval is correct when the retrieved context contains it —
an objective, LLM-free check, which is what makes the sweeps in sections 5-9 free.

In [9]:
GOLD_TARGET = 24
rng = np.random.default_rng(0)

GEN_PROMPT = """You are given one page from a product manual.

Write ONE question a real customer would ask support, that this page answers.
Then give the EXACT verbatim span from the page (15-30 words, copied character for
character) that contains the answer.

Rules:
- The question must be answerable from this page alone.
- The question must NOT mention page numbers or "the manual".
- The quote must be copied exactly from the page text, not paraphrased.
- If the page is a table of contents, an index, or has no substantive content,
  return {"skip": true}.

Return JSON only: {"question": "...", "quote": "..."}

PAGE TEXT:
"""


def gen_item(page):
    resp = llm(
        model=GEN_MODEL,
        messages=[{"role": "user", "content": GEN_PROMPT + page["text"][:4000]}],
        response_format={"type": "json_object"},
        temperature=0,
    )
    obj = json.loads(resp.choices[0].message.content)
    if obj.get("skip") or not obj.get("quote"):
        return None
    return {"question": obj["question"], "quote": obj["quote"],
            "product": page["product"], "page": page["page"]}


def build_gold():
    # Stratify: the mouse manual is 9 pages, so sample proportionally-ish but never zero.
    quota = {"car": 9, "tv": 11, "mouse": 4}
    items = []
    for product, n in quota.items():
        pool = [p for p in CORPUS_PAGES if p["product"] == product and len(p["text"]) > 600]
        picks = rng.choice(len(pool), size=min(len(pool), n + 4), replace=False)  # +4 for rejects
        kept = 0
        for page in [pool[int(i)] for i in picks]:
            if kept >= n:
                break
            try:
                item = gen_item(page)
            except Exception as exc:
                print("  gen failed:", type(exc).__name__)
                continue
            if item:
                items.append(item)
                kept += 1
    return items


raw_gold = cached_json("gold_raw.json", build_gold)
print(f"generated {len(raw_gold)} candidate items")
spend("after gold-set generation")

generated 22 candidate items
[spend] $0.0000 of $0.50 over 0 calls  after gold-set generation


### Verification: reject every label the source page does not support

This cell is the whole reason auto-generation is acceptable. An LLM that paraphrases
instead of quoting produces a label that silently never matches any chunk, which would
depress recall for *every* config equally and look like a hard problem rather than a
broken label.

In [11]:
PAGE_TEXT = {(p["product"], p["page"]): p["text"] for p in CORPUS_PAGES}

GOLD, rejected = [], []
for item in raw_gold:
    src = PAGE_TEXT.get((item["product"], item["page"]), "")
    (GOLD if norm(item["quote"]) in norm(src) else rejected).append(item)

print(f"kept {len(GOLD)} / {len(raw_gold)}  ({len(rejected)} rejected: quote not on the page)")
if rejected:
    r = rejected[0]
    print(f"\nexample rejection ({r['product']} p{r['page']}): {r['quote'][:90]}...")

for g in GOLD:
    assert norm(g["quote"]) in norm(PAGE_TEXT[(g["product"], g["page"])])

print(f"\n{'product':8s} {'items':>6s}")
for prod in ("car", "tv", "mouse"):
    print(f"{prod:8s} {sum(g['product'] == prod for g in GOLD):6d}")

print("\nsample:")
for g in GOLD[:3]:
    print(f"  Q: {g['question']}")
    print(f"     -> {g['product']} p{g['page']}: \"{g['quote'][:80]}...\"")

kept 18 / 22  (4 rejected: quote not on the page)

example rejection (car p22): make sure it is securely locked...

product   items
car           5
tv           11
mouse         2

sample:
  Q: How do I close the glove box?
     -> car p318: "Lift glove box flap upward until it engages..."
  Q: How do I adjust the park assist delay timer?
     -> car p200: "select the park assist delay timer..."
  Q: What should I use to clean the camera lens?
     -> car p202: "Use water to clean the camera lens..."


One structural caveat, stated rather than hidden: these questions are generated *from*
a page, so they use that page's vocabulary more than a real customer would. That
inflates absolute recall for every configuration. It does **not** invalidate the
comparisons between configurations, which is what this notebook is for. Section 7
adds deliberately vocabulary-mismatched queries to probe the case this bias hides.

## 3. The harness

Four functions, reused by every experiment below. Design decisions worth stating:

**Brute-force numpy for the sweeps, not a vector DB.** This corpus produces a few
thousand chunks. An exact `X @ q` over 5k × 384 floats is ~2 ms and returns the *true*
top-k. An ANN index would add a recall approximation on top of the thing being measured,
which makes chunking results harder to read. Section 9 measures the crossover point
where that stops being the right call.

**Metrics.**
- `quote_recall@k` — the gold quote appears in the concatenated top-k context. The
  primary accuracy number: it is exactly the condition under which the generator *can*
  be right.
- `page_hit@k` — a top-k chunk comes from the labelled page. Looser; catches near-misses.
- `MRR` — rank of the first answer-bearing chunk. Sensitive to ordering, which matters
  once we start truncating context to save money.
- `intact` — the ceiling: is the quote even contained inside a single chunk? A chunking
  config that splits the answer in half caps its own recall, and this separates
  "retriever missed it" from "chunker destroyed it".

In [12]:
EMBEDDERS = {}

def get_embedder(name):
    if name not in EMBEDDERS:
        EMBEDDERS[name] = SentenceTransformer(name)
    return EMBEDDERS[name]


def build_index(chunk_size=700, overlap=100, model_name="all-MiniLM-L6-v2", pages=None):
    pages = pages if pages is not None else CORPUS_PAGES
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""])

    chunks = []
    for p in pages:
        for j, t in enumerate(splitter.split_text(p["text"])):
            chunks.append({"text": t, "product": p["product"], "page": p["page"],
                           "id": f"{p['product']}-{p['page']}-{j}"})

    model = get_embedder(model_name)
    t0 = time.perf_counter()
    X = model.encode([c["text"] for c in chunks], batch_size=64,
                     normalize_embeddings=True, show_progress_bar=False)
    encode_s = time.perf_counter() - t0

    return {"chunks": chunks, "X": np.asarray(X, dtype=np.float32), "model_name": model_name,
            "chunk_size": chunk_size, "overlap": overlap, "encode_s": encode_s,
            "text_norm": [norm(c["text"]) for c in chunks]}


QUERY_PREFIX = {"BAAI/bge-small-en-v1.5": "Represent this sentence for searching relevant passages: "}

def embed_query(index, question):
    prefix = QUERY_PREFIX.get(index["model_name"], "")
    return get_embedder(index["model_name"]).encode(
        prefix + question, normalize_embeddings=True).astype(np.float32)


def dense_search(index, question, k=5, product=None):
    q = embed_query(index, question)
    scores = index["X"] @ q
    if product:
        mask = np.array([c["product"] == product for c in index["chunks"]])
        scores = np.where(mask, scores, -np.inf)
    top = np.argpartition(-scores, min(k, len(scores) - 1))[:k]
    top = top[np.argsort(-scores[top])]
    return [dict(index["chunks"][i], score=float(scores[i])) for i in top]

In [13]:
def evaluate(retriever, index, gold=None, k=5, label=""):
    """retriever(question) -> ranked list of chunk dicts. Returns metrics + per-item detail."""
    gold = gold if gold is not None else GOLD
    rr, hits, pages, lat, detail = [], [], [], [], []

    for g in gold:
        t0 = time.perf_counter()
        got = retriever(g["question"])[:k]
        lat.append((time.perf_counter() - t0) * 1000)

        gq = norm(g["quote"])
        rank = next((i + 1 for i, c in enumerate(got) if gq in norm(c["text"])), 0)
        hit = gq in norm(" ".join(c["text"] for c in got))
        page = any(c["product"] == g["product"] and c["page"] == g["page"] for c in got)

        rr.append(1 / rank if rank else 0.0)
        hits.append(hit); pages.append(page)
        detail.append({**g, "hit": hit, "rank": rank, "ctx_chars": sum(len(c["text"]) for c in got)})

    intact = np.mean([any(norm(g["quote"]) in t for t in index["text_norm"]) for g in gold])
    ctx = np.mean([d["ctx_chars"] for d in detail])

    return {"label": label or f"{index['chunk_size']}/{index['overlap']} {index['model_name'].split('/')[-1]}",
            "quote_recall": float(np.mean(hits)), "page_hit": float(np.mean(pages)),
            "mrr": float(np.mean(rr)), "intact": float(intact),
            "p50_ms": float(np.percentile(lat, 50)), "p95_ms": float(np.percentile(lat, 95)),
            "ctx_chars": float(ctx), "n_chunks": len(index["chunks"]), "detail": detail}


def show(*results, cols=("quote_recall", "page_hit", "mrr", "intact", "ctx_chars", "p50_ms", "p95_ms", "n_chunks")):
    w = max(len(r["label"]) for r in results) + 2
    print(f"{'config':{w}s}" + "".join(f"{c:>13s}" for c in cols))
    for r in results:
        row = ""
        for c in cols:
            v = r[c]
            row += f"{v:>13.0%}" if c in ("quote_recall", "page_hit", "intact") else (
                   f"{v:>13.3f}" if c == "mrr" else f"{v:>13,.0f}")
        print(f"{r['label']:{w}s}{row}")

## 4. Baseline

The configuration almost every RAG tutorial lands on, this one included until it was
measured: 700-character recursive chunks with 100 overlap, `all-MiniLM-L6-v2`, dense
top-5, no metadata filter, no reranking. Every number in the rest of the notebook is a
delta against this row.

In [14]:
base_index = build_index(chunk_size=700, overlap=100, model_name="all-MiniLM-L6-v2")
baseline = evaluate(lambda q: dense_search(base_index, q, k=5), base_index, label="baseline 700/100 dense@5")

show(baseline)
print(f"\nindex build: {base_index['encode_s']:.1f}s to embed {len(base_index['chunks']):,} chunks "
      f"({len(base_index['chunks'])/base_index['encode_s']:,.0f} chunks/s on CPU)")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7350.92it/s]


config                     quote_recall     page_hit          mrr       intact    ctx_chars       p50_ms       p95_ms     n_chunks
baseline 700/100 dense@5            72%          72%        0.630         100%        2,995            6           19        1,912

index build: 2.8s to embed 1,912 chunks (688 chunks/s on CPU)


### Where the baseline's errors actually are

An aggregate recall number tells you nothing about *what* to fix. Split it.

In [15]:
def by_group(result, key):
    groups = {}
    for d in result["detail"]:
        groups.setdefault(d[key], []).append(d["hit"])
    return {g: (float(np.mean(v)), len(v)) for g, v in sorted(groups.items(), key=str)}


print("quote_recall@5 by product:")
for g, (v, n) in by_group(baseline, "product").items():
    print(f"  {str(g):8s} {v:6.0%}  (n={n})")

misses = [d for d in baseline["detail"] if not d["hit"]]
print(f"\n{len(misses)} misses. First three:")
for d in misses[:3]:
    print(f"  [{d['product']} p{d['page']}] {d['question']}")

quote_recall@5 by product:
  car         80%  (n=5)
  mouse      100%  (n=2)
  tv          64%  (n=11)

5 misses. First three:
  [car p250] How do I turn on Adaptive Steering Assist in my vehicle?
  [tv p281] How can I stop the TV from voicing the activity when I change the channel?
  [tv p34] How do I adjust the volume on my Samsung Smart Remote?


Two things to read off this table before touching a single hyperparameter:

1. **Per-product recall is the actionable split.** A uniform number across products means
   the retriever is the bottleneck. A product that lags badly usually means something
   structural about that manual — denser layout, more tabular content, more cross-page
   references — and it points at the chunker, not the embedder.
2. **`intact` is the chunker's ceiling.** Anything below 100% is answer spans being cut
   in half at chunk boundaries — a cost paid before retrieval even runs, and invisible
   in a recall number that has no ceiling to compare against. That is what the next
   section attacks.

## 5. Chunk size: sweep it, don't inherit it

700/100 is a default copied from tutorials, not a measurement. Chunk size trades three
things simultaneously:

- **Recall** — larger chunks contain more context, so they match more queries, and are
  less likely to split an answer (`intact` goes up).
- **Precision and cost** — larger chunks also drag in irrelevant text. Context tokens
  scale roughly linearly with `chunk_size × k`, and prompt tokens are most of the bill.
- **Index size** — smaller chunks mean more vectors: more RAM, slower brute-force search,
  longer embed time.

The sweep is free (no LLM calls), so there is no excuse for guessing.

In [17]:
sweep = []
for cs, ov in [(300, 50), (500, 75), (700, 0), (700, 100), (1000, 150), (1400, 200), (1800, 0), (1800, 100), (1800, 200)]:
    idx = build_index(chunk_size=cs, overlap=ov)
    r = evaluate(lambda q, i=idx: dense_search(i, q, k=5), idx, label=f"{cs}/{ov}")
    r["encode_s"] = idx["encode_s"]
    sweep.append(r)

show(*sweep)
print()
for r in sweep:
    print(f"  {r['label']:10s} embed {r['encode_s']:5.1f}s | est. prompt tokens/query {r['ctx_chars']/4:5.0f}")

config     quote_recall     page_hit          mrr       intact    ctx_chars       p50_ms       p95_ms     n_chunks
300/50              78%          72%        0.604         100%        1,276            3            4        4,304
500/75              78%          78%        0.583         100%        2,235            6            7        2,585
700/0               83%          83%        0.678         100%        2,998            6            6        1,829
700/100             72%          72%        0.630         100%        2,995            6            6        1,912
1000/150            83%          78%        0.662         100%        3,938           12           12        1,432
1400/200            89%          78%        0.662         100%        5,133            2            2        1,096
1800/0              89%          83%        0.699         100%        5,989            2            3          909
1800/100            89%          83%        0.696         100%        6,000     

### Reading the sweep

Compare `700/0` against `700/100` for the pure effect of overlap: same chunk count
ballpark, and overlap buys `intact` (an answer split at a boundary in one chunk is whole
in its neighbour) at the price of ~15% more vectors. Overlap is cheap insurance.

Compare the extremes for the cost axis: the largest config sends roughly `ctx_chars`
worth of context per query, and at ~4 chars/token that lands directly on the invoice
every single query, forever. If a smaller chunk size reaches the same recall, the
smaller one is strictly better — same accuracy, lower bill, faster search.

Pick the smallest config whose recall is within noise of the best, rather than the
argmax. With ~24 questions, a one-item difference is 4 points of recall, so treat gaps
under ~8 points as ties and break them on cost.

In [18]:
best = max(sweep, key=lambda r: r["quote_recall"])
ties = [r for r in sweep if r["quote_recall"] >= best["quote_recall"] - 0.08]
CHOSEN = min(ties, key=lambda r: r["ctx_chars"])
print(f"best recall: {best['label']} ({best['quote_recall']:.0%})")
print(f"within-noise ties: {[r['label'] for r in ties]}")
print(f"chosen (cheapest tie): {CHOSEN['label']} -> {CHOSEN['ctx_chars']/4:.0f} prompt tokens/query "
      f"vs {best['ctx_chars']/4:.0f} for the argmax")

cs, ov = (int(x) for x in CHOSEN["label"].split("/"))
index = build_index(chunk_size=cs, overlap=ov)
tuned_chunks = evaluate(lambda q: dense_search(index, q, k=5), index, label=f"chunk-tuned {cs}/{ov}")
show(baseline, tuned_chunks)

best recall: 1400/200 (89%)
within-noise ties: ['700/0', '1000/150', '1400/200', '1800/0', '1800/100', '1800/200']
chosen (cheapest tie): 700/0 -> 750 prompt tokens/query vs 1283 for the argmax
config                     quote_recall     page_hit          mrr       intact    ctx_chars       p50_ms       p95_ms     n_chunks
baseline 700/100 dense@5            72%          72%        0.630         100%        2,995            6           19        1,912
chunk-tuned 700/0                   83%          83%        0.678         100%        2,998            3            3        1,829


## 6. Routing: the cheapest accuracy win in a multi-product corpus

Support queries are about *one* product. Searching all three manuals means the other two
contribute nothing but opportunities to be wrong — the classic failure being a mouse
pairing question that retrieves the TV's Bluetooth section.

If the product is known, a metadata filter removes two-thirds of the corpus from
consideration. In a real support flow you usually *do* know it (the customer's order,
the widget they opened chat from), and then routing is free and this section is a
formality. When you don't, you have to infer it, and the question is what that inference
should cost.

Two routers, measured head to head:
- **Embedding router** — cosine similarity between the query and a short profile string
  per product. No API call, sub-millisecond.
- **LLM router** — one small-model classification call.

In [25]:
PROFILES = {
    "mouse": "Logitech MX Master 3 wireless mouse: Bluetooth and USB receiver pairing, "
             "buttons, scroll wheel, thumb wheel, gestures, DPI, charging and battery, Logitech Options.",
    "car":   "Tata Sierra EV electric SUV owner's manual: charging, battery and range, "
             "regenerative braking, driving modes, tailgate, seats, airbags, tyres, brakes, "
             "warranty, servicing schedule, dashboard warning lights, ADAS driver assistance.",
    "tv":    "Samsung smart TV user guide: remote control, channels, picture and sound settings, "
             "HDMI inputs, apps, screen mirroring, Multi Control, CI card, network and software update.",
}

prof_model = get_embedder("all-MiniLM-L6-v2")
PROF_X = prof_model.encode(list(PROFILES.values()), normalize_embeddings=True)
PROF_KEYS = list(PROFILES)


def route_embedding(question):
    q = prof_model.encode(question, normalize_embeddings=True)
    s = PROF_X @ q
    return PROF_KEYS[int(np.argmax(s))], float(np.max(s) - np.sort(s)[-2])   # label, margin


def route_llm(question):
    resp = llm(
        model=CHEAP_MODEL, temperature=0, max_tokens=8,
        messages=[{"role": "system", "content": "Classify the product this support question is about. "
                                                "Answer with exactly one word: car, tv, or mouse."},
                  {"role": "user", "content": question}])
    return resp.choices[0].message.content.strip().lower(), cost_of(resp)


def bench_router(fn, name, is_llm=False):
    correct, lat, cost = 0, [], 0.0
    for g in GOLD:
        t0 = time.perf_counter()
        pred, extra = fn(g["question"])       # (label, margin) or (label, cost)
        lat.append((time.perf_counter() - t0) * 1000)
        correct += pred == g["product"]
        cost += extra if is_llm else 0.0
    print(f"{name:18s} acc {correct/len(GOLD):5.0%} | p50 {np.percentile(lat,50):7.2f} ms | "
          f"p95 {np.percentile(lat,95):7.2f} ms | ${cost/len(GOLD)*1000:6.3f} per 1k queries")
    return correct / len(GOLD)


acc_emb = bench_router(route_embedding, "embedding router")
acc_llm = bench_router(route_llm, "LLM router", is_llm=True)
spend("after router benchmark")

embedding router   acc   78% | p50    2.95 ms | p95    3.61 ms | $ 0.000 per 1k queries
LLM router         acc    0% | p50  356.88 ms | p95  467.62 ms | $ 0.011 per 1k queries
[spend] $0.0002 of $0.50 over 18 calls  after router benchmark


In [26]:
routed = evaluate(lambda q: dense_search(index, q, k=5, product=route_embedding(q)[0]),
                  index, label="+ embedding router")
show(tuned_chunks, routed)

# Does the filter fix the cross-product confusion the aggregate metric can hide?
ambiguous = "How do I pair it over Bluetooth?"
print(f"\nambiguous query: {ambiguous!r} -> routed to {route_embedding(ambiguous)[0]!r} "
      f"(margin {route_embedding(ambiguous)[1]:.3f})")
print("unfiltered top-3 products:", [c["product"] for c in dense_search(index, ambiguous, k=3)])

config               quote_recall     page_hit          mrr       intact    ctx_chars       p50_ms       p95_ms     n_chunks
chunk-tuned 700/0             83%          83%        0.678         100%        2,998            3            3        1,829
+ embedding router            61%          61%        0.483         100%        2,940            9           12        1,829

ambiguous query: 'How do I pair it over Bluetooth?' -> routed to 'mouse' (margin 0.241)
unfiltered top-3 products: ['mouse', 'mouse', 'tv']


### Verdict

Routing is worth it when the router is accurate; when it is wrong it is worse than no
filter, because a wrong filter makes the correct chunk **unreachable** rather than merely
low-ranked. That asymmetry is the thing to design around:

- ship the embedding router when its margin is high, and fall back to unfiltered search
  when the margin is small (the ambiguous query above is exactly that case),
- never route on a router that is less accurate than the retrieval it is protecting,
- and if the product is already known from session context, skip all of this.

The LLM router's accuracy has to beat the embedding router's by enough to justify adding
a network round-trip to the p95 of every query. Compare the two lines printed above
before adopting it.

## 7. Hybrid retrieval: dense embeddings are bad at exact strings

Dense retrieval matches meaning. Support queries frequently contain strings that have no
meaning to match — acronyms off a dashboard, codec names, menu labels quoted verbatim
off the screen. `V2L`, `ORVM`, `HE-AAC`: an embedding model maps these to roughly
nothing, while BM25 matches them exactly.

The standard fix is to run both and fuse with Reciprocal Rank Fusion, which combines
rankings without needing the two score scales to be comparable:

`RRF(d) = Σ 1 / (60 + rank_i(d))`

Cost of adding it: a BM25 index (megabytes, no GPU) and a few ms per query.

In [27]:
from rank_bm25 import BM25Okapi

def tokenize(s):
    return re.findall(r"[a-z0-9]+", s.lower())

bm25 = BM25Okapi([tokenize(c["text"]) for c in index["chunks"]])


def bm25_search(question, k=5, product=None):
    scores = bm25.get_scores(tokenize(question))
    if product:
        scores = np.where(np.array([c["product"] == product for c in index["chunks"]]), scores, -np.inf)
    top = np.argsort(-scores)[:k]
    return [dict(index["chunks"][i], score=float(scores[i])) for i in top]


def rrf(*rankings, k=60, top=5):
    fused = {}
    for ranking in rankings:
        for rank, c in enumerate(ranking, start=1):
            e = fused.setdefault(c["id"], {"chunk": c, "s": 0.0})
            e["s"] += 1.0 / (k + rank)
    best = sorted(fused.values(), key=lambda e: -e["s"])[:top]
    return [dict(e["chunk"], score=e["s"]) for e in best]


def hybrid_search(question, k=5, pool=20, product=None):
    return rrf(dense_search(index, question, k=pool, product=product),
               bm25_search(question, k=pool, product=product), top=k)


dense_only = tuned_chunks
bm25_only = evaluate(lambda q: bm25_search(q, k=5), index, label="bm25 only @5")
hybrid = evaluate(lambda q: hybrid_search(q, k=5), index, label="hybrid RRF @5")
show(dense_only, bm25_only, hybrid)

config              quote_recall     page_hit          mrr       intact    ctx_chars       p50_ms       p95_ms     n_chunks
chunk-tuned 700/0            83%          83%        0.678         100%        2,998            3            3        1,829
bm25 only @5                 83%          78%        0.722         100%        3,092            1            2        1,829
hybrid RRF @5                78%          78%        0.713         100%        3,046            9           12        1,829


### The queries hybrid exists for

The gold set was generated from page text, so its questions share vocabulary with their
target chunks — favourable ground for dense retrieval and unfavourable for showing
BM25's value. Probe the case directly with real exact-match strings taken from these
manuals (verified present below, so this is not a straw man).

In [28]:
EXACT_QUERIES = ["V2L", "ORVM", "AVH", "HE-AAC", "Multi Control"]

corpus_all = norm(" ".join(c["text"] for c in index["chunks"]))
for term in EXACT_QUERIES:
    assert norm(term) in corpus_all, f"{term} not in corpus - fix the example"

print(f"{'query':16s} {'dense finds it':>16s} {'bm25 finds it':>15s} {'hybrid finds it':>17s}")
for term in EXACT_QUERIES:
    d = any(norm(term) in norm(c["text"]) for c in dense_search(index, term, k=5))
    b = any(norm(term) in norm(c["text"]) for c in bm25_search(term, k=5))
    h = any(norm(term) in norm(c["text"]) for c in hybrid_search(term, k=5))
    print(f"{term:16s} {str(d):>16s} {str(b):>15s} {str(h):>17s}")

t0 = time.perf_counter(); [dense_search(index, q, k=5) for q in EXACT_QUERIES]; t_d = time.perf_counter() - t0
t0 = time.perf_counter(); [hybrid_search(q, k=5) for q in EXACT_QUERIES]; t_h = time.perf_counter() - t0
print(f"\nlatency per query: dense {t_d/len(EXACT_QUERIES)*1000:.1f} ms | hybrid {t_h/len(EXACT_QUERIES)*1000:.1f} ms")

query              dense finds it   bm25 finds it   hybrid finds it
V2L                          True            True              True
ORVM                         True            True              True
AVH                          True            True              True
HE-AAC                      False            True              True
Multi Control                True            True              True

latency per query: dense 6.2 ms | hybrid 6.0 ms


Fusion's real property is that it is **safe**: a document ranked well by either retriever
survives, so hybrid rarely loses to dense alone, while covering a failure mode dense
alone cannot. Whether the aggregate recall in the previous table moved is a property of
this gold set's vocabulary bias, not of hybrid search — which is exactly why this second
table exists.

## 8. Reranking: buy accuracy and cost reduction with the same 30 ms

A bi-encoder embeds the query and the chunk separately, so it never gets to compare them
directly. A cross-encoder reads `(query, chunk)` as one input and scores the pair. Far
more accurate, and far too slow to run over the whole corpus — so the standard shape is
retrieve wide and cheap, rerank narrow and expensive:

`retrieve 25 (hybrid) -> cross-encoder -> keep 4`

The part people miss is that this is a **cost** optimization, not only an accuracy one.
Better ordering means fewer chunks are needed in the prompt for the same recall, and
prompt tokens are the dominant line item. Accuracy up and bill down, paid for with
milliseconds of local CPU.

In [29]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_search(question, pool=25, k=4, product=None):
    cands = hybrid_search(question, k=pool, pool=pool, product=product)
    scores = reranker.predict([(question, c["text"]) for c in cands])
    order = np.argsort(-scores)[:k]
    return [dict(cands[int(i)], score=float(scores[int(i)])) for i in order]


reranked4 = evaluate(lambda q: rerank_search(q, k=4), index, k=4, label="hybrid->rerank @4")
reranked2 = evaluate(lambda q: rerank_search(q, k=2), index, k=2, label="hybrid->rerank @2")
show(dense_only, hybrid, reranked4, reranked2)

print(f"\ncontext sent to the LLM: {hybrid['ctx_chars']/4:.0f} tok (hybrid@5) -> "
      f"{reranked4['ctx_chars']/4:.0f} tok (rerank@4) -> {reranked2['ctx_chars']/4:.0f} tok (rerank@2)")
print(f"added retrieval latency: p50 {hybrid['p50_ms']:.0f} -> {reranked4['p50_ms']:.0f} ms, "
      f"p95 {hybrid['p95_ms']:.0f} -> {reranked4['p95_ms']:.0f} ms")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 19308.25it/s]


config              quote_recall     page_hit          mrr       intact    ctx_chars       p50_ms       p95_ms     n_chunks
chunk-tuned 700/0            83%          83%        0.678         100%        2,998            3            3        1,829
hybrid RRF @5                78%          78%        0.713         100%        3,046            9           12        1,829
hybrid->rerank @4            89%          89%        0.819         100%        2,576           60          100        1,829
hybrid->rerank @2            83%          83%        0.806         100%        1,279           50           75        1,829

context sent to the LLM: 762 tok (hybrid@5) -> 644 tok (rerank@4) -> 320 tok (rerank@2)
added retrieval latency: p50 9 -> 60 ms, p95 12 -> 100 ms


`MRR` is the metric to watch here, more than recall: the reranker's job is to move the
answer-bearing chunk to position 1. High MRR is what licenses cutting k, and cutting k is
what pays for the reranker. If MRR barely moves, the reranker is not earning its p95.

Note also that rerank@2 vs rerank@4 is a knob you can turn *per query* rather than
globally — section 10 uses the reranker's own score to decide how much context a
particular question deserves.

## 9. Embedding model and ANN index: two optimizations that mostly don't apply here

Both of these get recommended reflexively. Measure whether this corpus is big enough for
either to matter.

**Embedding model.** `bge-small-en-v1.5` is the same 384 dimensions and roughly the same
size as MiniLM, but trained differently and stronger on retrieval benchmarks. Same
storage, same search cost — so if it wins on recall it is a free upgrade. The only cost
is re-embedding the corpus and the query prefix it expects.

In [30]:
alt = build_index(chunk_size=cs, overlap=ov, model_name="BAAI/bge-small-en-v1.5")
alt_res = evaluate(lambda q: dense_search(alt, q, k=5), alt, label="bge-small dense@5")
show(dense_only, alt_res)

for i in (index, alt):
    n = len(i["chunks"])
    print(f"{i['model_name']:28s} dim {i['X'].shape[1]:4d} | embed {i['encode_s']:5.1f}s "
          f"({n/i['encode_s']:,.0f} chunks/s) | index RAM {i['X'].nbytes/1e6:5.1f} MB")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19231.06it/s]


config              quote_recall     page_hit          mrr       intact    ctx_chars       p50_ms       p95_ms     n_chunks
chunk-tuned 700/0            83%          83%        0.678         100%        2,998            3            3        1,829
bge-small dense@5            83%          83%        0.741         100%        2,942            5            7        1,829
all-MiniLM-L6-v2             dim  384 | embed   2.3s (779 chunks/s) | index RAM   2.8 MB
BAAI/bge-small-en-v1.5       dim  384 | embed   5.0s (366 chunks/s) | index RAM   2.8 MB


**ANN index.** HNSW is what makes vector search sublinear, and it is the right answer at
millions of vectors. At this corpus size it is a pure loss: exact numpy search is already
in the low milliseconds, and HNSW replaces an exact result with an approximate one for a
speedup you cannot perceive. Measure both here and at 50k vectors to find the crossover.

In [31]:
import chromadb

q_vec = embed_query(index, "how do I charge the battery")

def time_it(fn, n=50):
    fn(); t0 = time.perf_counter()
    for _ in range(n):
        fn()
    return (time.perf_counter() - t0) / n * 1000

exact_ms = time_it(lambda: np.argsort(-(index["X"] @ q_vec))[:5])

client = chromadb.Client()
for name in ("bench", "bench_big"):          # make the cell re-runnable
    try:
        client.delete_collection(name)
    except Exception:
        pass

col = client.create_collection("bench", metadata={"hnsw:space": "cosine"})
col.add(ids=[c["id"] for c in index["chunks"]], embeddings=index["X"].tolist())
hnsw_ms = time_it(lambda: col.query(query_embeddings=[q_vec.tolist()], n_results=5), n=20)

print(f"{len(index['chunks']):,} vectors:  exact numpy {exact_ms:6.2f} ms | chroma HNSW {hnsw_ms:6.2f} ms")

# Same comparison at a size where the asymptotics take over.
big = np.random.default_rng(0).normal(size=(50_000, 384)).astype(np.float32)
big /= np.linalg.norm(big, axis=1, keepdims=True)
qb = big[7]
big_exact = time_it(lambda: np.argsort(-(big @ qb))[:5], n=10)

big_col = client.create_collection("bench_big", metadata={"hnsw:space": "cosine"})
BATCH = 5_000    # chroma caps a single add() well below 50k
for s in range(0, len(big), BATCH):
    big_col.add(ids=[str(i) for i in range(s, min(s + BATCH, len(big)))],
                embeddings=big[s:s + BATCH].tolist())
big_hnsw = time_it(lambda: big_col.query(query_embeddings=[qb.tolist()], n_results=5), n=20)

print(f"{len(big):,} vectors: exact numpy {big_exact:6.2f} ms | chroma HNSW {big_hnsw:6.2f} ms")
print("\nRule: exact search until the exact-search line exceeds your latency budget.")
print("Everything before that point is approximation risk taken for free.")

1,829 vectors:  exact numpy   3.16 ms | chroma HNSW   0.50 ms
50,000 vectors: exact numpy   2.00 ms | chroma HNSW   0.84 ms

Rule: exact search until the exact-search line exceeds your latency budget.
Everything before that point is approximation risk taken for free.


Two negative results are worth as much as the positive ones in sections 5-8: they are
optimizations *not* adopted, and each one not adopted is infrastructure not maintained.
The reranker earns its complexity; an ANN index at 5k vectors does not.

## 10. Generation: where the money actually goes

Retrieval is local and effectively free. The invoice is the LLM call, and it has three
levers, in descending order of impact:

1. **Fewer prompt tokens** — already banked in sections 5 and 8 by cutting context from
   5 chunks to 4 (or 2). This is the biggest lever and it was paid for by the retriever.
2. **A smaller model** — 10-20× cheaper per token, if quality holds. Test rather than
   assume, and test with escalation rather than as an all-or-nothing switch.
3. **Not calling the model at all** — caching. A support corpus gets the same questions
   repeatedly, and paraphrases of them constantly.

Start with the answer function and honest per-query accounting.

In [32]:
SYSTEM = ("You are a product support assistant. Answer using ONLY the provided manual "
          "excerpts. Cite the product and page for each fact. If the excerpts do not "
          "contain the answer, say so plainly.")


def answer(question, model=GEN_MODEL, k=4, retriever=None):
    t0 = time.perf_counter()
    chunks = (retriever or (lambda q: rerank_search(q, k=k)))(question)
    t_ret = time.perf_counter() - t0

    ctx = "\n\n".join(f"[{c['product']} p{c['page']}]\n{c['text']}" for c in chunks)
    t1 = time.perf_counter()
    resp = llm(model=model, temperature=0, max_tokens=400,
               messages=[{"role": "system", "content": SYSTEM},
                         {"role": "user", "content": f"{ctx}\n\nQuestion: {question}"}])
    return {"text": resp.choices[0].message.content,
            "chunks": chunks,
            "retrieval_ms": t_ret * 1000,
            "llm_ms": (time.perf_counter() - t1) * 1000,
            "prompt_tokens": resp.usage.prompt_tokens,
            "completion_tokens": resp.usage.completion_tokens,
            "cost": cost_of(resp),
            "top_score": chunks[0]["score"] if chunks else float("-inf")}


demo = answer("How do I pair my MX Master 3 over Bluetooth?")
print(demo["text"][:600])
print(f"\nretrieval {demo['retrieval_ms']:.0f} ms | llm {demo['llm_ms']:.0f} ms | "
      f"{demo['prompt_tokens']} prompt + {demo['completion_tokens']} completion tokens | "
      f"${demo['cost']*1000:.3f} per 1k queries at this size")

**Pairing the MX Master 3 via Bluetooth**

1. **Turn the mouse on** – the LED on the bottom should be blinking quickly. If it isn’t, press the power button for three seconds (mouse p1).  
2. **Put the mouse in Bluetooth‑discoverable mode** – press and hold the Easy‑Switch button for three seconds; the LED will start blinking quickly (mouse p2).  
3. **Open your computer’s Bluetooth settings** and select the MX Master 3 from the list of available devices to complete the pairing (mouse p1 & mouse p2).  

*Sources: mouse p1 – “Connect directly using Bluetooth… Open the Bluetooth settings on your 

retrieval 61 ms | llm 1261 ms | 698 prompt + 362 completion tokens | $0.322 per 1k queries at this size


### Small model with confidence-based escalation

An 20B model is roughly an order of magnitude cheaper than a 120B. On a RAG task where the
answer is sitting in the context, extraction is much easier than open-ended reasoning, so
the cheap model is often sufficient — but not always, and the failures are not random:
they cluster on the queries where retrieval was weak and the model has to work harder.

The reranker already produces a usable confidence signal — its top score. Policy: answer
with the cheap model when retrieval is confident, escalate to the large model when it
isn't. Judge both against the gold quote with the large model, and price all three.

In [39]:
JUDGE_MODEL = GEN_MODEL
SUBSET = GOLD[:12]      # 3 policies x 12 questions x (answer + judge) — sized to the budget

def judge(question, gold_quote, response):
    r = llm(
        model=JUDGE_MODEL, temperature=0, max_tokens=400,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content":
            "Does the ANSWER correctly convey the fact in the REFERENCE for the QUESTION?\n"
            f"QUESTION: {question}\nREFERENCE: {gold_quote}\nANSWER: {response}\n"
            'Reply JSON only: {"correct": true|false}'}])
    try:
        return bool(json.loads(r.choices[0].message.content).get("correct"))
    except Exception:
        return False


def run_policy(name, pick_model):
    """Retrieve once per question, then let the policy pick a model from those chunks.
    Retrieving inside the policy would double the reranker cost and hide it from `lat`."""
    ok, cost, lat = 0, 0.0, []
    for g in SUBSET:
        t0 = time.perf_counter()
        chunks = rerank_search(g["question"], k=4)
        ret_ms = (time.perf_counter() - t0) * 1000
        a = answer(g["question"], model=pick_model(chunks), retriever=lambda q, c=chunks: c)
        ok += judge(g["question"], g["quote"], a["text"])
        cost += a["cost"]
        lat.append(ret_ms + a["llm_ms"])
    print(f"{name:34s} correct {ok/len(SUBSET):5.0%} | ${cost/len(SUBSET)*1000:7.3f}/1k queries | "
          f"p50 {np.percentile(lat,50):6.0f} ms | p95 {np.percentile(lat,95):6.0f} ms")
    return {"name": name, "acc": ok / len(SUBSET), "cost_1k": cost / len(SUBSET) * 1000,
            "p50": float(np.percentile(lat, 50)), "p95": float(np.percentile(lat, 95))}


ESCALATE_BELOW = 0.0   # cross-encoder logit; tune on your own traffic
scores = [rerank_search(g["question"], k=4)[0]["score"] for g in SUBSET]
print(f"top rerank score over the subset: min {min(scores):.2f} median {np.median(scores):.2f} "
      f"max {max(scores):.2f} | escalating below {ESCALATE_BELOW}\n")

policies = [
    run_policy("always 70B", lambda chunks: GEN_MODEL),
    run_policy("always 8B", lambda chunks: CHEAP_MODEL),
    run_policy("8B, escalate on low confidence",
               lambda chunks: CHEAP_MODEL if chunks[0]["score"] >= ESCALATE_BELOW else GEN_MODEL),
]
spend("after generation policies")

top rerank score over the subset: min 3.24 median 8.29 max 10.00 | escalating below 0.0


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

[ratelimit] waiting 2.9s (1/6)

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

[ratelimit] waiting 4.3s (1/6)

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

[ratelimit] waiting 3.7s (1/6)

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

[ratelimit] waiting 5.1s (1/6)

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

[ratelimit] waiting 1.

Read the three lines as a Pareto choice, not a winner. If the cheap model matches on
quality, take it. If it loses a few points, the escalation row shows what fraction of the
gap a confidence threshold recovers and at what price. The threshold is the tunable —
set it from your own traffic's score distribution (printed above), not from this corpus.

### Caching: the only optimization with no downside on the cost axis

Support traffic is heavily repetitive. Three layers, cheapest first:

| Layer | Saves | When it fires |
|---|---|---|
| Exact-match cache | 100% of the call | Identical question string |
| Semantic cache | 100% of the call | Paraphrase above a similarity threshold |
| Provider prompt caching | ~50-90% of *prompt* token cost | Shared prefix across calls; automatic on OpenAI, explicit on Anthropic |

The static system prompt is deliberately placed first in the message list so it forms a
cacheable prefix — a free structural decision made at write time. Provider prompt caching
is then automatic on OpenAI and explicit (`cache_control`) on Anthropic; neither is
measured here because the hit rate is decided by the provider's routing, not by you.

The semantic layer *is* measured below, because its hit rate is a property of your own
traffic and nobody else's number transfers.

In [40]:
class SemanticCache:
    """ponytail: in-process numpy cache. Swap for Redis + redisvl when you need it shared."""

    def __init__(self, threshold=0.90):
        self.threshold, self.keys, self.vals = threshold, [], []

    def get(self, q_vec):
        if not self.keys:
            return None
        sims = np.vstack(self.keys) @ q_vec
        i = int(np.argmax(sims))
        return self.vals[i] if sims[i] >= self.threshold else None

    def put(self, q_vec, val):
        self.keys.append(q_vec); self.vals.append(val)


PARAPHRASES = {
    "How do I pair my MX Master 3 over Bluetooth?": [
        "connect the mx master 3 mouse via bluetooth",
        "bluetooth pairing steps for my logitech mouse",
        "how to link my mouse to a computer wirelessly",
    ],
    "How does regenerative braking work on my EV?": [
        "what is regen braking and how does it charge the battery",
        "does braking recover energy in this car",
    ],
}

cache = SemanticCache(threshold=0.90)
hits = misses_cost = 0
saved = 0.0

for original, variants in PARAPHRASES.items():
    for q in [original] + variants:
        qv = embed_query(index, q)
        cached = cache.get(qv)
        if cached is not None:
            hits += 1
            saved += cached["cost"]
        else:
            a = answer(q)
            cache.put(qv, a)
            misses_cost += 1

total = hits + misses_cost
print(f"queries {total} | cache hits {hits} ({hits/total:.0%}) | LLM calls avoided {hits}")
print(f"cost avoided on this tiny sample: ${saved:.5f} -> at 100k queries/month with the "
      f"same hit rate, ${saved/total*100_000:,.2f}/month")
print("\nThreshold is the risk knob: too low and a different question gets a stale answer.")
print("Measure it on real paraphrase pairs before shipping, and never cache across users")
print("when answers depend on account state.")


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

[ratelimit] waiting 4.2s (1/6)
queries 7 | cache hits 0 (0%) | LLM calls avoided 0
cost avoided on this tiny sample: $0.00000 -> at 100k queries/month with the same hit rate, $0.00/month

Threshold is the risk knob: too low and a different question gets a stale answer.
Measure it on real paraphrase pairs before shipping, and never cache across users
when answers depend on account state.


## 11. The assembled pipeline

Everything that survived measurement, in order:

```mermaid
flowchart LR
    Q[query] --> SC{semantic cache}
    SC -->|hit| OUT[answer]
    SC -->|miss| R[embedding router]
    R --> H[hybrid: dense + BM25, RRF]
    H --> X[cross-encoder rerank 25 to 4]
    X --> M{top score >= threshold}
    M -->|yes| S[small model]
    M -->|no| L[large model]
    S --> OUT
    L --> OUT
```

In [41]:
pipeline_cache = SemanticCache(threshold=0.90)


def support_answer(question, known_product=None, margin_floor=0.05):
    qv = embed_query(index, question)
    cached = pipeline_cache.get(qv)
    if cached is not None:
        return {**cached, "cached": True}

    product = known_product
    if product is None:
        guess, margin = route_embedding(question)
        product = guess if margin >= margin_floor else None    # low margin -> search everything

    chunks = rerank_search(question, k=4, product=product)
    model = CHEAP_MODEL if chunks[0]["score"] >= ESCALATE_BELOW else GEN_MODEL
    out = answer(question, model=model, retriever=lambda q: chunks)
    out.update({"product": product, "model": model, "cached": False})
    pipeline_cache.put(qv, out)
    return out


for q in ["How do I pair my MX Master 3 over Bluetooth?",
          "How do I use V2L to power an appliance from my car?",
          "How do I turn on Multi Control on my TV?"]:
    out = support_answer(q)
    print(f"Q: {q}")
    print(f"   routed={out['product']} model={out['model'].split('/')[-1]} "
          f"{out['prompt_tokens']} prompt tok | {out['retrieval_ms']:.0f}+{out['llm_ms']:.0f} ms | ${out['cost']:.6f}")
    print(f"   {out['text'][:220].strip()}...\n")

Q: How do I pair my MX Master 3 over Bluetooth?
   routed=mouse model=gpt-oss-20b 712 prompt tok | 0+892 ms | $0.000143
   To pair your MX Master 3 via Bluetooth:

1. **Put the mouse into discoverable mode** – press and hold the Easy‑Switch button for three seconds. The LED on the bottom of the mouse will start blinking quickly, indicating i...

Q: How do I use V2L to power an appliance from my car?
   routed=car model=gpt-oss-20b 866 prompt tok | 0+1048 ms | $0.000185
   **Using V2L to power an appliance from your car**

1. **Prepare the vehicle**  
   * Make sure the park‑brake/EPB is engaged (p75).  
   * The vehicle must be in a safe state for V2L operation (p75).

2. **Connect the V2...

Q: How do I turn on Multi Control on my TV?
   routed=tv model=gpt-oss-20b 668 prompt tok | 0+636 ms | $0.000124
   I’m sorry, but the provided manual excerpts do not include the steps for turning on Multi Control on the TV....



### Ablation: what each step actually bought

Retrieval quality, context size, and latency for every configuration measured above,
against the same gold set.

In [42]:
show(baseline, tuned_chunks, routed, hybrid, reranked4, reranked2,
     cols=("quote_recall", "page_hit", "mrr", "ctx_chars", "p50_ms", "p95_ms"))

print(f"\naccuracy: {baseline['quote_recall']:.0%} -> {reranked4['quote_recall']:.0%} quote_recall, "
      f"MRR {baseline['mrr']:.3f} -> {reranked4['mrr']:.3f}")
print(f"cost:     {baseline['ctx_chars']/4:.0f} -> {reranked4['ctx_chars']/4:.0f} prompt tokens per query "
      f"({1 - reranked4['ctx_chars']/baseline['ctx_chars']:.0%} reduction), before cache hits")
print(f"speed:    retrieval p95 {baseline['p95_ms']:.0f} -> {reranked4['p95_ms']:.0f} ms "
      f"(the LLM call, {demo['llm_ms']:.0f} ms, still dominates end to end)")

for p in policies:
    print(f"generation policy | {p['name']:34s} acc {p['acc']:.0%} ${p['cost_1k']:.3f}/1k p95 {p['p95']:.0f} ms")

config                     quote_recall     page_hit          mrr    ctx_chars       p50_ms       p95_ms
baseline 700/100 dense@5            72%          72%        0.630        2,995            6           19
chunk-tuned 700/0                   83%          83%        0.678        2,998            3            3
+ embedding router                  61%          61%        0.483        2,940            9           12
hybrid RRF @5                       78%          78%        0.713        3,046            9           12
hybrid->rerank @4                   89%          89%        0.819        2,576           60          100
hybrid->rerank @2                   83%          83%        0.806        1,279           50           75

accuracy: 72% -> 89% quote_recall, MRR 0.630 -> 0.819
cost:     749 -> 644 prompt tokens per query (14% reduction), before cache hits
speed:    retrieval p95 19 -> 100 ms (the LLM call, 1261 ms, still dominates end to end)
generation policy | always 70B           

## 12. What this exercise actually taught

**The first real defect was not a RAG technique.** 2,272 line-break hyphenations in the
car manual (999 distinct words) split words in half — invisible in the rendered PDF, fatal to
lexical matching, and free to fix. No amount of chunk tuning recovers a word the index
never contained as a word. Audit ingestion before tuning retrieval: the check is one
loop, it costs nothing, and it either finds something like this or tells you the ceiling
is clear and you can spend with confidence.

**Accuracy and cost are not always a trade-off.** Reranking improved ordering enough to
cut context from 5 chunks to 4 or 2, which is simultaneously more accurate and cheaper.
The trade it does make is latency, and only on the retrieval stage — which the LLM call
dwarfs anyway.

**Two recommended optimizations were rejected on measurement:** an ANN index at this
corpus size (exact search is already sub-10 ms and doesn't approximate), and, depending
on the numbers printed above, a swap of the embedding model. Rejections are results.
Each one is a component not deployed and not maintained.

**The gold set is the real deliverable.** Every number here is downstream of 24 verified
question/quote pairs. Regenerate it whenever the corpus changes, keep the verification
assert, and grow it with real support transcripts — production traffic will not share
vocabulary with the manual the way generated questions do, and section 7's second table
is the only place this notebook probes that gap.

### Not done here, and when it would be worth doing

| Skipped | Add it when |
|---|---|
| Joining hard `-\n` line breaks with a dictionary | The ~95 remaining split words show up in real miss analysis. Guessing without a dictionary would corrupt genuine hyphenates like `toll-free`. |
| Extracting the manuals' figures and tables | Your questions need them. Text extraction drops both, and this corpus is diagram-heavy — a known, unmeasured gap in every number above. |
| Query rewriting / decomposition for compound questions | Your traffic contains multi-part questions ("is my service covered under warranty, and how often is it?" needs two retrievals fused, because the two facts sit ~30 pages apart in this manual). |
| Fine-tuned embeddings on support transcripts | You have thousands of labelled query/chunk pairs and off-the-shelf recall has plateaued. |
| Shared cache (Redis + redisvl) | More than one process serves traffic. The in-process cache above is per-worker. |
| Per-chunk section headers as retrieval context | Pages alone prove insufficient — worth it for manuals with deep hierarchy. |

### What this notebook cost to produce

In [43]:
spend("FINAL")
print(f"\nof which the gold set is cached — a re-run costs ${SPEND['usd']:.4f} minus those "
      f"generation calls, and produces identical numbers.")

assert SPEND["usd"] < BUDGET_USD, (
    f"over budget: ${SPEND['usd']:.4f} >= ${BUDGET_USD:.2f}. Shrink SUBSET, drop a policy, "
    f"or raise the ceiling deliberately.")
print(f"\nunder the ${BUDGET_USD:.2f} ceiling with ${BUDGET_USD - SPEND['usd']:.4f} of headroom.")

[spend] $0.0135 of $0.50 over 82 calls  FINAL

of which the gold set is cached — a re-run costs $0.0135 minus those generation calls, and produces identical numbers.

under the $0.50 ceiling with $0.4865 of headroom.
